In [1]:
import polars as pl
import os

print("🔍 EXPLORAÇÃO PRÉ-JOGO - Versão Corrigida (raw StatsBomb)\n")

df = pl.read_parquet("statsbomb_completo.parquet")

print(f"📊 Total de linhas: {df.height:,}")
print(f"📋 Colunas disponíveis: {len(df.columns)}")
print("Exemplos de colunas:", df.columns[:15])  # só para confirmar

# ====================== TOP 20 JOGADORES ======================
top_players = (
    df.group_by("player.name")
    .agg([
        pl.len().alias("total_eventos"),
        pl.col("match_id").n_unique().alias("partidas_jogadas")
    ])
    .sort("total_eventos", descending=True)
    .head(20)
)
print("\n🏆 TOP 20 JOGADORES COM MAIS DADOS")
print(top_players)

# ====================== TOP 15 TIMES ======================
top_teams = (
    df.group_by("team.name")
    .agg([
        pl.len().alias("total_eventos"),
        pl.col("match_id").n_unique().alias("partidas")
    ])
    .sort("total_eventos", descending=True)
    .head(15)
)
print("\n🏟️ TOP 15 TIMES COM MAIS DADOS")
print(top_teams)

# ====================== ESTATÍSTICAS DE FINALIZAÇÃO ======================
print("\n⚽ Processando chutes (pode demorar alguns segundos)...")

shots = df.filter(pl.col("type.name") == "Shot")

# Criamos is_goal na hora
player_shots = (
    shots.group_by("player.name")
    .agg([
        pl.sum("shot.statsbomb_xg").alias("xg_total"),
        (pl.col("shot.outcome.name") == "Goal").cast(pl.Int32).sum().alias("gols_reais"),
        pl.len().alias("chutes"),
        pl.col("match_id").n_unique().alias("partidas_com_chute")
    ])
    .filter(pl.col("chutes") >= 5)
    .sort("xg_total", descending=True)
    .head(20)
)
print("\n⚽ TOP 20 JOGADORES POR xG TOTAL (mínimo 5 chutes)")
print(player_shots)

# Bônus: jogadores com mais gols reais
print("\n🥇 TOP 10 JOGADORES POR GOLS REAIS")
top_gols = (
    player_shots.sort("gols_reais", descending=True)
    .head(10)
)
print(top_gols)

print("\n" + "="*90)
print("✅ Pronto! Copie e cole **todo o output** aqui (as 4 tabelas).")
print("Com isso eu monto imediatamente o pipeline completo de pré-jogo.")

🔍 EXPLORAÇÃO PRÉ-JOGO - Versão Corrigida (raw StatsBomb)

📊 Total de linhas: 12,188,949
📋 Colunas disponíveis: 150
Exemplos de colunas: ['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type.id', 'type.name', 'possession_team.id', 'possession_team.name', 'play_pattern.id', 'play_pattern.name', 'team.id']

🏆 TOP 20 JOGADORES COM MAIS DADOS
shape: (20, 3)
┌────────────────────────────────┬───────────────┬──────────────────┐
│ player.name                    ┆ total_eventos ┆ partidas_jogadas │
│ ---                            ┆ ---           ┆ ---              │
│ str                            ┆ u32           ┆ u32              │
╞════════════════════════════════╪═══════════════╪══════════════════╡
│ Lionel Andrés Messi Cuccittini ┆ 133405        ┆ 603              │
│ Sergio Busquets i Burgos       ┆ 90437         ┆ 407              │
│ Xavier Hernández Creus         ┆ 73222         ┆ 265              │
│ Andrés Iniesta Luján           ┆ 72875        

: 

In [ ]:
import polars as pl
import os

print("🔍 EXPLORAÇÃO PRÉ-JOGO: Jogadores e Times com mais dados\n")

# Carrega o arquivo principal (use o completo se possível)
df = pl.read_parquet("statsbomb_completo.parquet")

print(f"📊 Total de linhas: {df.height:,}")
print("📋 Colunas disponíveis:")
print(df.columns)

# Detecta coluna de time automaticamente
team_col = None
for possible in ["team_name", "team", "team_id", "Squad"]:
    if possible in df.columns:
        team_col = possible
        break

print(f"🏷️ Coluna de time encontrada: {team_col or 'NENHUMA'}")

# TOP 20 JOGADORES com mais dados
top_players = (
    df.group_by("player_name")
    .agg([
        pl.count().alias("total_eventos"),
        pl.col("match_id").n_unique().alias("partidas_jogadas")
    ])
    .sort("total_eventos", descending=True)
    .head(20)
)
print("\n🏆 TOP 20 JOGADORES COM MAIS DADOS")
print(top_players)

# Se tiver coluna de time → TOP TIMES
if team_col:
    top_teams = (
        df.group_by(team_col)
        .agg([
            pl.count().alias("total_eventos"),
            pl.col("match_id").n_unique().alias("partidas")
        ])
        .sort("total_eventos", descending=True)
        .head(15)
    )
    print("\n🏟️ TOP 15 TIMES COM MAIS DADOS")
    print(top_teams)

# Estatísticas de finalização/gols (mais relevante para prever gols)
if "type_name" in df.columns and "shot_statsbomb_xg" in df.columns:
    shots = df.filter(pl.col("type_name") == "Shot")
    
    player_shots = (
        shots.group_by("player_name")
        .agg([
            pl.sum("shot_statsbomb_xg").alias("xg_total"),
            pl.col("is_goal").sum().alias("gols_reais") if "is_goal" in df.columns else pl.lit(0).alias("gols_reais"),
            pl.count().alias("chutes"),
            pl.col("match_id").n_unique().alias("partidas_com_chute")
        ])
        .filter(pl.col("chutes") >= 5)  # mínimo para confiabilidade
        .sort("xg_total", descending=True)
        .head(20)
    )
    print("\n⚽ TOP 20 JOGADORES POR xG TOTAL (com pelo menos 5 chutes)")
    print(player_shots)
else:
    print("\n⚠️ Não encontrei colunas de Shot/xG. Vamos trabalhar com o que tem.")

print("\n" + "="*80)
print("✅ Rode isso e cole aqui o output completo (especialmente as tabelas de top jogadores e times).")
print("Com isso eu vejo exatamente quem tem dados suficientes e monto o pipeline de pré-jogo na sequência.")

🔍 EXPLORAÇÃO PRÉ-JOGO: Jogadores e Times com mais dados

📊 Total de linhas: 12,188,949
📋 Colunas disponíveis:
['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type.id', 'type.name', 'possession_team.id', 'possession_team.name', 'play_pattern.id', 'play_pattern.name', 'team.id', 'team.name', 'tactics.formation', 'tactics.lineup', 'related_events', 'location', 'player.id', 'player.name', 'position.id', 'position.name', 'pass.recipient.id', 'pass.recipient.name', 'pass.length', 'pass.angle', 'pass.height.id', 'pass.height.name', 'pass.end_location', 'pass.body_part.id', 'pass.body_part.name', 'pass.type.id', 'pass.type.name', 'carry.end_location', 'pass.switch', 'pass.outcome.id', 'pass.outcome.name', 'ball_receipt.outcome.id', 'ball_receipt.outcome.name', 'under_pressure', 'duel.type.id', 'duel.type.name', 'pass.aerial_won', 'counterpress', 'interception.outcome.id', 'interception.outcome.name', 'off_camera', 'ball_recovery.recovery_failure', 'pass.as

C:\Users\kaiki\AppData\Local\Temp\ipykernel_7732\1016532306.py:26: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_eventos"),


ColumnNotFoundError: unable to find column "player_name"; valid columns: ["id", "index", "period", "timestamp", "minute", "second", "possession", "duration", "type.id", "type.name", "possession_team.id", "possession_team.name", "play_pattern.id", "play_pattern.name", "team.id", "team.name", "tactics.formation", "tactics.lineup", "related_events", "location", "player.id", "player.name", "position.id", "position.name", "pass.recipient.id", "pass.recipient.name", "pass.length", "pass.angle", "pass.height.id", "pass.height.name", "pass.end_location", "pass.body_part.id", "pass.body_part.name", "pass.type.id", "pass.type.name", "carry.end_location", "pass.switch", "pass.outcome.id", "pass.outcome.name", "ball_receipt.outcome.id", "ball_receipt.outcome.name", "under_pressure", "duel.type.id", "duel.type.name", "pass.aerial_won", "counterpress", "interception.outcome.id", "interception.outcome.name", "off_camera", "ball_recovery.recovery_failure", "pass.assisted_shot_id", "pass.shot_assist", 